# Mervin/Mervis -- Gemma 4 E2B LoRA fine-tune (Google Colab)

LoRA fine-tunes **Gemma 4 E2B-it** (the smaller sibling of the arena's Gemma 4
E4B) on the Mervin/Mervis persona dataset with
[Unsloth](https://github.com/unslothai/unsloth), exports a 4-bit **Q4_K_M
GGUF** (~3 GB), and uploads it to Hugging Face -- where `serve.py`
auto-downloads it like the other models.

**No system prompt.** The Mervin/Mervis tags and behavior come *purely* from
fine-tuning: training data is bare `user -> assistant`, so the model produces
the format with or without any system prompt (the direction we want for every
arena model).

Trains on Google Colab. A GPU is required -- a free T4 is plenty for E2B with
Unsloth 4-bit; A100/L4 are faster.

### Before you run
- Runtime -> Change runtime type -> **GPU**.
- Add a Colab **secret** `HF_TOKEN` (key icon, left sidebar) with a Hugging
  Face **write** token. The upload cell reads it via `userdata`; it is never
  written into the notebook.

### Gemma 4 + PEFT note
Gemma 4 wraps its attention/MLP projections in `Gemma4ClippableLinear`, which
older PEFT does not recognize. We pin **peft >= 0.19.0** (adds Gemma 4 default
targets); if you hit a "target modules not found" error on an older PEFT, pass
the regex `r".*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)\.linear"`
as `target_modules` instead.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "|", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

In [ ]:
# Unsloth + a dependency set it supports; drop the runtime's broken torchao
# (incompatible with the bleeding-edge torch here; optional for bnb 4-bit).
# peft>=0.19.0 is required so PEFT recognizes Gemma 4's Gemma4ClippableLinear.
!pip install --upgrade -q unsloth unsloth_zoo
!pip install -q "transformers==4.56.2" "datasets>=3.4.1,<4.4.0" "trl>=0.18.2,<=0.24.0,!=0.19.0" "peft>=0.19.0"
!pip uninstall -q -y torchao
import transformers, trl, datasets, peft
print("transformers", transformers.__version__, "| trl", trl.__version__,
      "| datasets", datasets.__version__, "| peft", peft.__version__)

In [ ]:
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/gemma-4-E2B-it",   # not gated; google/gemma-4-E2B-it also works
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,
    load_in_4bit    = True,
    full_finetuning = False,
)
print("loaded gemma-4-E2B-it")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = 32,
    lora_dropout               = 0.05,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 3407,
)
model.print_trainable_parameters()

## Dataset -- no system prompt

Each example is just `user -> assistant` rendered with Gemma's native chat
template (`<start_of_turn>user ... <end_of_turn><start_of_turn>model ...`). The
persona is carried entirely by the assistant targets, so the fine-tune -- not a
system prompt -- is what makes the model speak as Mervin/Mervis.

In [ ]:
import csv, io, urllib.request
from datasets import Dataset

CSV_URL = "https://raw.githubusercontent.com/CarlFreeAiEngineer/merv/main/mervin_mervis_finetune.csv"

raw  = urllib.request.urlopen(CSV_URL).read().decode("utf-8")
rows = list(csv.DictReader(io.StringIO(raw)))
print(f"Loaded {len(rows)} rows")

def build_prompt(user_msg):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user_msg}],
        tokenize=False, add_generation_prompt=True)

def to_text(r):
    return {"text": build_prompt(r["prompt"]) + " " + r["response"] + tokenizer.eos_token}

ds = Dataset.from_list([to_text(r) for r in rows])
print(ds[0]["text"][:500])

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = ds,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LEN,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 3,
        learning_rate               = 2e-4,
        warmup_ratio                = 0.1,
        lr_scheduler_type           = "cosine",
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        logging_steps               = 5,
        seed                        = 3407,
        output_dir                  = "outputs",
        report_to                   = "none",
    ),
)
trainer.train()

## Sanity check -- bare user message, no system prompt

In [ ]:
FastLanguageModel.for_inference(model)

def ask(q):
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, top_p=0.9)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

for q in ["What is 2+2?", "Tell me about Mondays."]:
    print("Q:", q); print(ask(q)); print("-"*60)

## Export Q4_K_M GGUF

We quantize **before** upload and push only the quantized file (~3 GB).

In [ ]:
import glob, os, time
t0 = time.time()
model.save_pretrained_gguf("gemma-merv-gguf", tokenizer, quantization_method="q4_k_m")
print("export done in", round(time.time()-t0), "s")
for f in sorted(glob.glob("/content/**/*Q4_K_M.gguf", recursive=True)):
    print(round(os.path.getsize(f)/1e9, 2), "GB ", f)

## Upload directly to Hugging Face

Pushes the quantized GGUF to `freeideas/merv-gemma4e2b` as `model-q4_k_m.gguf`,
where `serve.py` auto-downloads it. The token comes from the Colab `HF_TOKEN`
secret -- never written into the notebook.

In [ ]:
import glob
from google.colab import userdata
from huggingface_hub import HfApi

tok  = userdata.get("HF_TOKEN")          # write token, set in Colab Secrets
repo = "freeideas/merv-gemma4e2b"
src  = glob.glob("/content/**/*Q4_K_M.gguf", recursive=True)[0]

api = HfApi()
print("uploading as model-q4_k_m.gguf ->", repo, "( as", api.whoami(token=tok)["name"], ")")
api.create_repo(repo, repo_type="model", exist_ok=True, token=tok)
api.upload_file(path_or_fileobj=src, path_in_repo="model-q4_k_m.gguf",
                repo_id=repo, repo_type="model", token=tok)
print("done -> https://huggingface.co/" + repo)

## In the arena

`serve.py` and `index.html` carry a `gemma2b` entry, so any host running
`serve.py` auto-downloads `model-q4_k_m.gguf` from `freeideas/merv-gemma4e2b` on
startup and the model appears in the dropdown. See this folder's `README.md`.